# Source-Cited Web Research Brief Agent (PydanticAI + OpenRouter)
**Kunal Ahuja** | Automate with GenAI & Agents | Module 5 Graded Assignment

A conversational ReAct agent that decides when to search DuckDuckGo, read Wikipedia, and fetch and clean web pages, then writes a 5-8 paragraph research brief with inline numeric citations `[n]`, at least two short quotes (10 words or fewer) and a References section.

**Tools:** `ddg_search`, `wiki_summary`, `fetch_page_clean`, `select_top_urls`, `save_logs`.
**Conversation:** one shared message history, so follow-up questions build on earlier work and citation numbers stay stable.
**Evaluation:** automatic checks write `scorecard.json`. Quotes are verified against the text the agent actually fetched.
**Outputs:** `brief.md`, `report.json`, `scorecard.json`, `logs/tool_logs.json` (all in `research_outputs/`, also zipped).

The API key is read from Colab Secrets (`OPENROUTER_API_KEY`) or a hidden prompt and is never stored in the notebook.

## Cell 1: Install, imports and secure API key

In [1]:
%pip install -q "pydantic-ai-slim[openai]==2.62.0" ddgs beautifulsoup4 requests

import asyncio
import json
import os
import re
import shutil
import threading
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from getpass import getpass
from urllib.parse import quote, urlparse

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.usage import UsageLimits

try:
    from google.colab import userdata
    api_key = userdata.get("OPENROUTER_API_KEY")   # Colab Secrets (key icon)
except Exception:
    api_key = None
if not api_key:
    api_key = getpass("Enter your OpenRouter API key: ")

api_key = (api_key or "").strip()            # drop stray spaces/newlines from pasting
if not api_key:
    raise ValueError("No API key found. Add a Colab secret named OPENROUTER_API_KEY "
                     "(Notebook access ON) and re-run this cell.")
print(f"API key loaded ({len(api_key)} characters, "
      f"{'looks like an OpenRouter key' if api_key.startswith('sk-or-') else 'does NOT start with sk-or-, check it'})")

Enter your OpenRouter API key: ··········
API key loaded (73 characters, looks like an OpenRouter key)


## Cell 2: Configuration (topic, model, follow-up questions)

In [2]:
LLM_MODEL = "openai/gpt-4o-mini"     # must support tool calling; change the slug to try others
TOPIC = "Solid-state batteries for electric vehicles: promise and obstacles"

# Follow-up questions used to demonstrate conversational state (Cell 9).
FOLLOW_UPS = [
    "Which single obstacle do your sources suggest is the most serious? Answer in 3 sentences with citations.",
    "In one short paragraph, what should a car buyer take away from this? Use only your sources.",
]

OUT_DIR = "research_outputs"         # brief.md, report.json, scorecard.json, logs/
LOG_PATH = os.path.join(OUT_DIR, "logs", "tool_logs.json")
MAX_PAGE_CHARS = 4000                # cleaned text kept per page (what the model sees)
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; Module5ResearchBot/1.0; student project)"}
os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)

## Cell 3: Shared research state (citation registry and audit trail)

In [3]:
def domain_of(url: str) -> str:
    """Return the bare domain of a URL (no 'www.')."""
    host = urlparse(url).netloc.lower()
    return host[4:] if host.startswith("www.") else host

@dataclass
class ResearchState:
    """Shared state for one research session. It is passed to every tool through
    RunContext and persists across follow-up turns, so citation numbers stay stable."""
    queries: list = field(default_factory=list)          # every search query issued
    search_results: list = field(default_factory=list)   # raw DuckDuckGo results
    selected_urls: list = field(default_factory=list)    # output of select_top_urls
    sources: dict = field(default_factory=dict)          # id -> {id,title,url,domain,kind,text}
    url_to_id: dict = field(default_factory=dict)
    tool_calls: list = field(default_factory=list)       # audit trail of every tool call
    lock: threading.Lock = field(default_factory=threading.Lock)

    def log(self, tool: str, args: dict, summary: str) -> None:
        with self.lock:
            self.tool_calls.append({"time": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                                    "tool": tool, "args": args, "summary": summary})

    def register_source(self, title: str, url: str, text: str, kind: str) -> int:
        """Give each fetched page a stable citation number (1, 2, 3, ...)."""
        with self.lock:
            if url in self.url_to_id:
                return self.url_to_id[url]
            sid = len(self.sources) + 1
            self.sources[sid] = {"id": sid, "title": title, "url": url,
                                 "domain": domain_of(url), "kind": kind, "text": text}
            self.url_to_id[url] = sid
            return sid

state = ResearchState()

## Cell 4: Agent definition and instructions

In [11]:
INSTRUCTIONS = """You are a careful research assistant that writes source-grounded briefs.
Work in a ReAct loop: decide what information you need, call a tool, read the result, repeat.

WORKFLOW (follow every step)
1. Call ddg_search 1-3 times with different queries.
2. Call wiki_summary once for background on the main topic.
3. Call select_top_urls to choose the best candidates, then call fetch_page_clean on at least 4
   of them, covering at least 3 different domains. Many sites block bots: if a fetch fails or
   returns too little text, immediately try another URL (search again if you run out).
4. Call save_logs once before writing your final answer.
5. Write the brief using ONLY facts that appear in tool results. Do not use outside knowledge.
   If the sources do not support a claim, leave it out.

CITATION RULES
- Cite with numeric markers like [1], [2]. The number is the "Source N" returned by
  fetch_page_clean or wiki_summary. Never invent a number. Every paragraph needs a citation.
  Write separate brackets such as [1][2], never [1, 2].
- Draw on at least 3 different sources in the brief and cite each one you fetched.

QUOTE RULES (required)
- Include at least two short direct quotes in double quotes. Each quote must be at most
  10 words and copied exactly, word for word, from the fetched text, followed by its citation.
  Example: Researchers note that "interface resistance remains a problem" [3].

FORMAT
- A '# Title' line, then 5-8 paragraphs, then a '## References' section with one line per
  cited source in the form: [n] Source title - URL

FOLLOW-UP QUESTIONS
- Answer from the sources already gathered. Search or fetch more only if needed. Keep citation
  numbers consistent, keep answers short, and do not rewrite the whole brief unless asked."""

model = OpenAIChatModel(
    LLM_MODEL,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1", api_key=api_key),
)

agent = Agent(
    model,
    deps_type=ResearchState,
    instructions=INSTRUCTIONS,
    model_settings={"temperature": 0.2},
)
print("Agent ready:", LLM_MODEL)

Agent ready: openai/gpt-4o-mini


## Cell 5: Tools (ddg_search, select_top_urls, fetch_page_clean, wiki_summary, save_logs)

In [12]:
# ---------------------------------------------------------------- ddg_search
@agent.tool
def ddg_search(ctx: RunContext[ResearchState], query: str, max_results: int = 6) -> str:
    """Search the web with DuckDuckGo. Returns numbered results with title, URL and snippet."""
    try:
        from ddgs import DDGS
    except ImportError:
        from duckduckgo_search import DDGS
    results, last_err = [], None
    for attempt in range(3):                              # retry: DDG sometimes rate-limits
        try:
            results = list(DDGS().text(query, max_results=max_results))
            if results:
                break
        except Exception as e:
            last_err = e
            time.sleep(2 * (attempt + 1))
    st = ctx.deps
    st.queries.append(query)
    rows = []
    for r in results:
        item = {"query": query, "title": r.get("title", ""), "url": r.get("href", ""),
                "snippet": r.get("body", "")}
        if item["url"]:
            rows.append(item)
    with st.lock:
        st.search_results.extend(rows)
    st.log("ddg_search", {"query": query}, f"{len(rows)} results" + (f" (error: {last_err})" if last_err and not rows else ""))
    if not rows:
        return f"No results (error: {last_err}). Try a different query."
    return "\n".join(f"{i+1}. {r['title']} | {r['url']} | {r['snippet'][:200]}" for i, r in enumerate(rows))

# ------------------------------------------------------------ select_top_urls
LOW_VALUE = ("pinterest.", "facebook.", "instagram.", "tiktok.", "youtube.", "youtu.be",
             "twitter.com", "x.com", "linkedin.com", "quora.com")

@agent.tool
def select_top_urls(ctx: RunContext[ResearchState], k: int = 6) -> str:
    """Pick the k most useful, domain-diverse candidate URLs from all search results so far."""
    st = ctx.deps
    terms = {w for q in st.queries for w in re.findall(r"[a-z0-9]{4,}", q.lower())}
    seen, scored = set(), []
    for r in st.search_results:
        if r["url"] in seen or any(bad in r["url"].lower() for bad in LOW_VALUE):
            continue
        seen.add(r["url"])
        text = (r["title"] + " " + r["snippet"]).lower()
        scored.append((sum(t in text for t in terms), r))      # keyword-overlap relevance
    scored.sort(key=lambda x: -x[0])
    chosen, used_domains = [], set()
    for _, r in scored:                                        # pass 1: one URL per domain
        if domain_of(r["url"]) not in used_domains and len(chosen) < k:
            chosen.append(r); used_domains.add(domain_of(r["url"]))
    for _, r in scored:                                        # pass 2: fill remaining slots
        if r not in chosen and len(chosen) < k:
            chosen.append(r)
    st.selected_urls = [{"title": r["title"], "url": r["url"], "domain": domain_of(r["url"])} for r in chosen]
    st.log("select_top_urls", {"k": k}, f"selected {len(chosen)} URLs from {len(used_domains)} domains")
    if not chosen:
        return "No candidate URLs yet. Call ddg_search first."
    return "\n".join(f"{i+1}. {c['title']} | {c['url']}" for i, c in enumerate(st.selected_urls))

# ----------------------------------------------------------- fetch_page_clean
NOISE_TAGS = ["script", "style", "noscript", "nav", "footer", "header", "aside", "form",
              "svg", "iframe", "button", "figure", "dialog"]
NOISE_ATTR = re.compile(r"cookie|banner|advert|sidebar|menu|popup|newsletter|subscribe|share|social|comment", re.I)

def clean_html(html: str) -> tuple:
    """Strip page chrome and return (title, readable text). Keeps headings, paragraphs and list items."""
    soup = BeautifulSoup(html, "html.parser")
    title = (soup.title.string or "").strip() if soup.title and soup.title.string else ""
    for tag in soup(NOISE_TAGS):
        tag.decompose()
    for tag in soup.find_all(attrs={"class": NOISE_ATTR}) + soup.find_all(attrs={"id": NOISE_ATTR}):
        # Never delete structural wrappers: sites often put classes like "has-sidebar" on <body>
        # or on a div that contains the whole article.
        if getattr(tag, "decomposed", False) or tag.name in ("html", "body", "main", "article") \
                or tag.find(["article", "main"]):
            continue
        tag.decompose()
    root = soup.find("article") or soup.find("main") or soup.body or soup
    lines, seen = [], set()
    for el in root.find_all(["h1", "h2", "h3", "p", "li"]):
        t = " ".join(el.get_text(" ", strip=True).split())
        if len(t) < (20 if el.name.startswith("h") else 50) or t in seen:   # drop short/duplicate noise
            continue
        seen.add(t); lines.append(t)
    return title, "\n".join(lines)

@agent.tool
def fetch_page_clean(ctx: RunContext[ResearchState], url: str) -> str:
    """Download a web page, remove HTML noise, and return clean text labelled with its Source number."""
    st = ctx.deps
    try:
        resp = requests.get(url, headers=HEADERS, timeout=12)
        resp.raise_for_status()
        title, text = clean_html(resp.text)
    except Exception as e:
        st.log("fetch_page_clean", {"url": url}, f"FAILED: {e}")
        return f"Could not fetch {url}: {e}. Try a different URL."
    text = text[:MAX_PAGE_CHARS]
    if len(text) < 200:
        st.log("fetch_page_clean", {"url": url}, "too little text after cleaning")
        return f"Page {url} had too little readable text. Try a different URL."
    sid = st.register_source(title or domain_of(url), url, text, "web")
    st.log("fetch_page_clean", {"url": url}, f"Source {sid}, {len(text)} chars")
    return f"[Source {sid}] {title or domain_of(url)} ({url})\n{text}"

# ---------------------------------------------------------------- wiki_summary
@agent.tool
def wiki_summary(ctx: RunContext[ResearchState], topic: str) -> str:
    """Get the Wikipedia summary for a topic using the Wikipedia REST API."""
    st = ctx.deps
    try:
        found = requests.get("https://en.wikipedia.org/w/rest.php/v1/search/page",
                             params={"q": topic, "limit": 1}, headers=HEADERS, timeout=12).json()
        pages = found.get("pages", [])
        if not pages:
            st.log("wiki_summary", {"topic": topic}, "no article found")
            return f"No Wikipedia article found for '{topic}'."
        key = pages[0]["key"]
        s = requests.get(f"https://en.wikipedia.org/api/rest_v1/page/summary/{quote(key)}",
                         headers=HEADERS, timeout=12).json()
        url = s.get("content_urls", {}).get("desktop", {}).get("page") or f"https://en.wikipedia.org/wiki/{quote(key)}"
        text = s.get("extract", "")[:MAX_PAGE_CHARS]
    except Exception as e:
        st.log("wiki_summary", {"topic": topic}, f"FAILED: {e}")
        return f"Wikipedia lookup failed: {e}"
    sid = st.register_source(f"Wikipedia: {s.get('title', key)}", url, text, "wikipedia")
    st.log("wiki_summary", {"topic": topic}, f"Source {sid}, {len(text)} chars")
    return f"[Source {sid}] Wikipedia: {s.get('title', key)} ({url})\n{text}"

# ------------------------------------------------------------------- save_logs
def write_logs(st: ResearchState) -> str:
    """Persist search results, selected URLs and snippets (used for grading)."""
    snippets = [{"source_id": s["id"], "url": s["url"], "snippet": s["text"][:300]}
                for s in st.sources.values()]
    payload = {"queries": st.queries, "search_results": st.search_results,
               "selected_urls": st.selected_urls, "snippets": snippets,
               "tool_calls": st.tool_calls}
    with open(LOG_PATH, "w") as f:
        json.dump(payload, f, indent=2)
    return LOG_PATH

@agent.tool
def save_logs(ctx: RunContext[ResearchState]) -> str:
    """Save the search results, selected URLs and snippets to disk. Call once before the final answer."""
    path = write_logs(ctx.deps)
    ctx.deps.log("save_logs", {}, f"wrote {path}")
    return f"Logs saved to {path}"

print("Registered tools:", ["ddg_search", "wiki_summary", "fetch_page_clean", "select_top_urls", "save_logs"])

Registered tools: ['ddg_search', 'wiki_summary', 'fetch_page_clean', 'select_top_urls', 'save_logs']


## Cell 6: Conversational turn runner (keeps message history)

In [13]:
history = []        # message history shared across turns (conversational memory)
turn_log = []       # per-turn latency/usage for report.json

async def run_turn(prompt: str) -> str:
    """Run one conversational turn. The agent decides which tools to call (ReAct), and the
    full message history is passed back in so follow-ups can build on earlier work."""
    start = time.time()
    result = await agent.run(prompt, deps=state, message_history=history,
                             usage_limits=UsageLimits(request_limit=40))
    history[:] = result.all_messages()                       # preserve history between turns
    usage = result.usage() if callable(result.usage) else result.usage   # property in newer versions
    turn_log.append({"prompt": prompt, "latency_sec": round(time.time() - start, 2),
                     "requests": usage.requests, "tool_calls": usage.tool_calls,
                     "input_tokens": usage.input_tokens, "output_tokens": usage.output_tokens})
    return result.output

## Cell 7: Evaluation (quote verification, citation checks, scorecard) and output writers

In [14]:
def split_brief(text: str) -> tuple:
    """Split a brief into (body, references_block). References block is '' if absent."""
    m = re.search(r"(?im)^\s*#{0,6}\s*\**references\**\s*:?\s*$", text)
    return (text[:m.start()].rstrip(), text[m.start():]) if m else (text.rstrip(), "")

def norm(s: str) -> str:
    """Lowercase and strip punctuation so quote matching ignores formatting differences."""
    return " ".join(re.sub(r"[^a-z0-9]+", " ", s.lower().replace("’", "'")).split())

def analyse_brief(raw: str) -> dict:
    """Check the agent's brief against the sources it actually fetched."""
    body, refs_block = split_brief(raw)
    cited = sorted({int(n) for grp in re.findall(r"\[(\d+(?:\s*,\s*\d+)*)\]", body)
                    for n in re.split(r"\s*,\s*", grp)})        # handles [1], [1][2] and [1, 2]
    unresolved = [n for n in cited if n not in state.sources]
    quotes = []
    for a, b in re.findall(r"“([^”]+)”|\"([^\"]+)\"", body):
        q = (a or b).strip()
        words = len(q.split())
        if words < 3:
            continue
        hit = [sid for sid, s in state.sources.items() if norm(q) in norm(s["text"])]
        quotes.append({"text": q, "words": words, "verified": bool(hit), "source_ids": hit})
    paragraphs = [p for p in re.split(r"\n\s*\n", body)
                  if not p.strip().startswith("#") and len(p.split()) >= 25]
    # Rebuild References from the tool registry so every [n] maps to the true title + URL.
    ref_lines = [f"[{n}] {state.sources[n]['title']} - {state.sources[n]['url']}"
                 for n in cited if n in state.sources]
    final = body + "\n\n## References\n\n" + "\n\n".join(ref_lines) + "\n"   # blank lines so Markdown keeps one per line
    return {"final": final, "body": body, "cited": cited, "unresolved": unresolved,
            "quotes": quotes, "paragraphs": len(paragraphs),
            "agent_wrote_references": bool(refs_block.strip())}

def build_scorecard(a: dict) -> dict:
    """Automatic checks from the assignment, written to scorecard.json."""
    domains = sorted({s["domain"] for s in state.sources.values()})
    cited_domains = sorted({state.sources[n]["domain"] for n in a["cited"] if n in state.sources})
    good_quotes = [q for q in a["quotes"] if q["verified"] and q["words"] <= 10]
    long_quotes = [q for q in a["quotes"] if q["words"] > 10]
    log_ok = False
    if os.path.exists(LOG_PATH) and os.path.getsize(LOG_PATH) > 0:
        with open(LOG_PATH) as f:
            lg = json.load(f)
        log_ok = all(lg.get(k) for k in ("search_results", "selected_urls", "snippets"))
    checks = {
        "unique_domains": {"required": 3, "found_in_research": len(domains), "domains": domains,
                           "cited_domains": cited_domains,
                           "pass": len(cited_domains) >= 3},          # brief must actually use 3+ domains
        "short_quotes": {"required": 2, "verified_short_quotes": len(good_quotes),
                         "quotes_over_10_words": len(long_quotes),
                         "pass": len(good_quotes) >= 2 and not long_quotes},
        "references_section": {"present": "## References" in a["final"] and bool(a["cited"]),
                               "agent_wrote_it_itself": a["agent_wrote_references"],
                               "pass": "## References" in a["final"] and bool(a["cited"])},
        "citations_resolve": {"cited": a["cited"], "unresolved": a["unresolved"],
                              "pass": bool(a["cited"]) and not a["unresolved"]},
        "paragraph_count": {"required": "5-8", "found": a["paragraphs"],
                            "pass": 5 <= a["paragraphs"] <= 8},
        "logs_created": {"path": LOG_PATH, "pass": log_ok},
    }
    passed = sum(c["pass"] for c in checks.values())
    return {"topic": TOPIC, "checks": checks, "passed": passed, "total": len(checks),
            "overall_pass": passed == len(checks)}

def save_outputs(raw_brief: str, follow_ups: list) -> dict:
    """Write brief.md, report.json, scorecard.json and the tool logs."""
    a = analyse_brief(raw_brief)
    write_logs(state)
    with open(os.path.join(OUT_DIR, "brief.md"), "w") as f:
        f.write(a["final"])
    counts = {}
    for c in state.tool_calls:
        counts[c["tool"]] = counts.get(c["tool"], 0) + 1
    report = {"topic": TOPIC, "model": LLM_MODEL,
              "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
              "turns": turn_log, "tool_call_counts": counts,
              "sources": [{k: v for k, v in s.items() if k != "text"} for s in state.sources.values()],
              "citations_used": a["cited"], "quotes": a["quotes"],
              "brief_word_count": len(a["body"].split()),
              "agent_wrote_references_section": a["agent_wrote_references"],
              "follow_ups": follow_ups}
    with open(os.path.join(OUT_DIR, "report.json"), "w") as f:
        json.dump(report, f, indent=2)
    score = build_scorecard(a)
    with open(os.path.join(OUT_DIR, "scorecard.json"), "w") as f:
        json.dump(score, f, indent=2)
    return {"analysis": a, "scorecard": score}


def quote_candidates(limit: int = 6) -> list:
    """Exact short phrases (<=10 words) copied from fetched pages. Used to help the agent
    fix a brief that is missing verified quotes."""
    per_source = []
    for sid, src in state.sources.items():
        found = []
        for sent in re.split(r"(?<=[.!?])\s+", src["text"]):
            if not 70 <= len(sent) <= 220 or "[" in sent:
                continue
            text = " ".join(sent.split()[:9])
            cut = max(text.rfind(","), text.rfind(";"))
            text = (text[:cut] if cut > 25 else text).strip(" ,;:.")
            if len(text.split()) >= 4 and norm(text) in norm(src["text"]):
                found.append((sid, text))
            if len(found) == 2:
                break
        per_source.append(found)
    out = []
    for i in range(2):                      # round-robin so quotes come from different sources
        for found in per_source:
            if i < len(found) and len(out) < limit:
                out.append(found[i])
    return out

async def refine_brief(raw: str, max_rounds: int = 2) -> str:
    """Evaluate-and-revise loop: run the automatic checks on the draft, tell the agent exactly
    what failed, and ask for a revised brief (the agent keeps its conversation history)."""
    for rnd in range(1, max_rounds + 1):
        a = analyse_brief(raw)
        problems = []
        good = [q for q in a["quotes"] if q["verified"] and q["words"] <= 10]
        if len(good) < 2:
            cands = "; ".join(f'"{t}" (Source {sid})' for sid, t in quote_candidates())
            problems.append("It needs at least two short direct quotes (max 10 words, in double "
                            "quotes, each followed by a [n] citation). You may quote these exact "
                            f"phrases from your sources: {cands}")
        if any(q["words"] > 10 for q in a["quotes"]):
            problems.append("Some quotes are longer than 10 words. Shorten them.")
        cited_domains = {state.sources[n]["domain"] for n in a["cited"] if n in state.sources}
        if len({s["domain"] for s in state.sources.values()}) < 3:
            problems.append(f"Only {len({s['domain'] for s in state.sources.values()})} website(s) have been fetched. "
                            "Use the tools to fetch pages from at least 3 different domains before writing.")
        elif len(cited_domains) < 3:
            listing = ", ".join(f"[{s['id']}] {s['title']}" for s in state.sources.values())
            problems.append(f"It cites only {len(cited_domains)} website(s). Cite at least 3 different "
                            f"sources. Available sources: {listing}")
        if a["unresolved"]:
            problems.append(f"Citations {a['unresolved']} do not match any fetched source. Remove or fix them.")
        if not 5 <= a["paragraphs"] <= 8:
            problems.append(f"It must have 5-8 paragraphs (it has {a['paragraphs']}).")
        if not problems:
            print(f"Refinement: draft passed all checks after {rnd - 1} revision(s).")
            return raw
        print(f"Refinement round {rnd}: asking the agent to fix {len(problems)} problem(s):")
        for p in problems:
            print("  -", p[:160])
        raw = await run_turn("Revise your brief. The automatic checks found these problems:\n- "
                             + "\n- ".join(problems)
                             + "\nReturn the complete revised brief in the same format (# title, 5-8 "
                               "paragraphs with [n] citations, ## References).")
    return raw

## Cell 8: Run the research agent on the topic

In [15]:
print(f"Researching: {TOPIC}\n")
raw_brief = await run_turn(f"Research topic: {TOPIC}\nResearch it with your tools and write the brief.")
raw_brief = await refine_brief(raw_brief)       # check the draft and let the agent fix problems
print("\nTool calls so far:")
for c in state.tool_calls:
    print(f"  {c['tool']:17s} {c['summary']}")
display(Markdown(raw_brief))

Researching: Solid-state batteries for electric vehicles: promise and obstacles

Refinement round 1: asking the agent to fix 2 problem(s):
  - It needs at least two short direct quotes (max 10 words, in double quotes, each followed by a [n] citation). You may quote these exact phrases from your sources
  - Some quotes are longer than 10 words. Shorten them.
Refinement: draft passed all checks after 1 revision(s).

Tool calls so far:
  ddg_search        6 results
  select_top_urls   selected 4 URLs from 4 domains
  fetch_page_clean  FAILED: 403 Client Error: Forbidden for url: https://www.researchgate.net/publication/381819006_Challenges_and_Advancements_in_All-Solid-State_Battery_Technology_for_Electric_Vehicles
  fetch_page_clean  FAILED: 403 Client Error: Forbidden for url: https://www.mdpi.com/2571-8800/7/3/12
  fetch_page_clean  too little text after cleaning
  fetch_page_clean  Source 1, 4000 chars
  ddg_search        6 results
  select_top_urls   selected 4 URLs from 4 domains
  

# Solid-State Batteries for Electric Vehicles: Promise and Obstacles

Solid-state batteries (SSBs) represent a significant advancement in battery technology, particularly for electric vehicles (EVs). A solid-state battery (SSB) is an electrical battery that uses a solid electrolyte to conduct ions between the electrodes, instead of the liquid or gel polymer electrolytes found in conventional batteries [4]. This shift could lead to "higher energy density" and improved safety profiles [2]. However, while the potential benefits are substantial, several technical and economic challenges remain that hinder widespread adoption.

One of the most compelling advantages of solid-state batteries is their safety. Traditional lithium-ion batteries are prone to thermal runaway, a condition that can lead to fires or explosions under certain circumstances. "Solid-State Batteries eliminate these risks through inherently safe solid" electrolytes, which can withstand higher temperatures and are less likely to decompose under stress [3]. Furthermore, the solid structure of these batteries helps prevent lithium dendrite growth, a common cause of short circuits in liquid batteries. This inherent safety makes SSBs particularly attractive for applications in electric vehicles, where safety is paramount.

In terms of performance, solid-state batteries promise exceptional energy density. They can potentially exceed the energy density limits of liquid batteries, which are currently capped at around 300 Wh/kg. For electric vehicles, this could mean "higher energy density, faster charging" [2]. SSBs can support high-capacity materials, such as lithium metal anodes, which can store significantly more lithium per unit mass than conventional graphite anodes [2][5]. This capability could translate into longer driving ranges for EVs, addressing one of the major consumer concerns regarding electric vehicle adoption.

Despite these advantages, solid-state batteries face significant hurdles before they can be commercially viable. Key challenges include low ionic conductivity and poor interface stability. Solid electrolytes often exhibit high interface resistance, which can impede lithium-ion transport and lead to performance degradation over time [3][5]. Additionally, the manufacturing processes for SSBs are not yet optimized for large-scale production, raising concerns about cost and durability. As noted, "For several years now, solid-state batteries have been touted" as the next big step forward for electric vehicles, but their viability remains uncertain [5].

The timeline for commercialization of solid-state batteries is still evolving. Major automotive companies and battery manufacturers are actively investing in research and development, with some targeting production readiness by 2027-2028 [2]. However, early adopters may only see these batteries in premium EV models initially, as the technology is still transitioning from prototype to mass-market applications [2]. 

In conclusion, while solid-state batteries hold great promise for revolutionizing electric vehicle technology through enhanced safety and performance, significant obstacles must be overcome. The path to widespread adoption will require continued innovation in materials science, manufacturing processes, and cost reduction strategies.

## References
[1] Wikipedia: Solid-state battery - https://en.wikipedia.org/wiki/Solid-state_battery  
[2] Solid-State Batteries: The Future of Electric Vehicles - https://electricvehicletalks.com/solid-state-batteries/  
[3] Solid-State Batteries: Advantages, Challenges, and Roadmap - https://atomfair.com/battery-research/solid-state-batteries-advantages-challenges/  
[4] Pros And Cons Of Solid-State Batteries: Are Solid-State Batteries The Savior of The EV? - https://carbuzz.com/solid-state-batteries-pros-cons  
[5] Solid-State Batteries: EV Range, Safety & 2026 Outlook - https://electricvehicletalks.com/solid-state-batteries/  

## Cell 9: Follow-up questions (conversation memory)

In [16]:
follow_up_records = []
for q in FOLLOW_UPS:
    print(f"\nFOLLOW-UP: {q}")
    answer = await run_turn(q)          # history carries over from the previous turn
    follow_up_records.append({"question": q, "answer": answer})
    display(Markdown(answer))


FOLLOW-UP: Which single obstacle do your sources suggest is the most serious? Answer in 3 sentences with citations.


The most serious obstacle facing solid-state batteries is their low ionic conductivity and poor interface stability. Solid electrolytes often exhibit high interface resistance, which can impede lithium-ion transport and lead to performance degradation over time [3][5]. This issue significantly hinders the overall efficiency and reliability of solid-state batteries, making it a critical barrier to their commercialization in electric vehicles.


FOLLOW-UP: In one short paragraph, what should a car buyer take away from this? Use only your sources.


Car buyers should be aware that while solid-state batteries promise significant advantages for electric vehicles, such as improved safety and higher energy density, they are still in the developmental phase. Major manufacturers are targeting production readiness by 2027-2028, but challenges like low ionic conductivity and interface stability remain significant hurdles [2][5]. Therefore, potential buyers may initially see solid-state batteries in premium models, with widespread availability in mainstream vehicles still a few years away.

## Cell 10: Save deliverables and show the scorecard

In [17]:
outputs = save_outputs(raw_brief, follow_up_records)
print("=== FINAL BRIEF (References rebuilt from the tool registry) ===")
display(Markdown(outputs["analysis"]["final"]))
print("\n=== SCORECARD ===")
print(json.dumps(outputs["scorecard"], indent=2))

shutil.make_archive("research_outputs", "zip", OUT_DIR)      # one zip with all deliverables
print("\nFiles written:", sorted(os.listdir(OUT_DIR)), "+ logs/tool_logs.json, research_outputs.zip")
try:
    from google.colab import files
    files.download("research_outputs.zip")                    # downloads to your computer
except Exception:
    pass

=== FINAL BRIEF (References rebuilt from the tool registry) ===


# Solid-State Batteries for Electric Vehicles: Promise and Obstacles

Solid-state batteries (SSBs) represent a significant advancement in battery technology, particularly for electric vehicles (EVs). A solid-state battery (SSB) is an electrical battery that uses a solid electrolyte to conduct ions between the electrodes, instead of the liquid or gel polymer electrolytes found in conventional batteries [4]. This shift could lead to "higher energy density" and improved safety profiles [2]. However, while the potential benefits are substantial, several technical and economic challenges remain that hinder widespread adoption.

One of the most compelling advantages of solid-state batteries is their safety. Traditional lithium-ion batteries are prone to thermal runaway, a condition that can lead to fires or explosions under certain circumstances. "Solid-State Batteries eliminate these risks through inherently safe solid" electrolytes, which can withstand higher temperatures and are less likely to decompose under stress [3]. Furthermore, the solid structure of these batteries helps prevent lithium dendrite growth, a common cause of short circuits in liquid batteries. This inherent safety makes SSBs particularly attractive for applications in electric vehicles, where safety is paramount.

In terms of performance, solid-state batteries promise exceptional energy density. They can potentially exceed the energy density limits of liquid batteries, which are currently capped at around 300 Wh/kg. For electric vehicles, this could mean "higher energy density, faster charging" [2]. SSBs can support high-capacity materials, such as lithium metal anodes, which can store significantly more lithium per unit mass than conventional graphite anodes [2][5]. This capability could translate into longer driving ranges for EVs, addressing one of the major consumer concerns regarding electric vehicle adoption.

Despite these advantages, solid-state batteries face significant hurdles before they can be commercially viable. Key challenges include low ionic conductivity and poor interface stability. Solid electrolytes often exhibit high interface resistance, which can impede lithium-ion transport and lead to performance degradation over time [3][5]. Additionally, the manufacturing processes for SSBs are not yet optimized for large-scale production, raising concerns about cost and durability. As noted, "For several years now, solid-state batteries have been touted" as the next big step forward for electric vehicles, but their viability remains uncertain [5].

The timeline for commercialization of solid-state batteries is still evolving. Major automotive companies and battery manufacturers are actively investing in research and development, with some targeting production readiness by 2027-2028 [2]. However, early adopters may only see these batteries in premium EV models initially, as the technology is still transitioning from prototype to mass-market applications [2]. 

In conclusion, while solid-state batteries hold great promise for revolutionizing electric vehicle technology through enhanced safety and performance, significant obstacles must be overcome. The path to widespread adoption will require continued innovation in materials science, manufacturing processes, and cost reduction strategies.

## References

[2] Solid-State Batteries: The Future of Electric Vehicles - https://electricvehicletalks.com/solid-state-batteries/

[3] Solid-State Batteries: Advantages, Challenges, and Roadmap - https://atomfair.com/battery-research/solid-state-batteries-advantages-challenges/

[4] Wikipedia: Solid-state battery - https://en.wikipedia.org/wiki/Solid-state_battery

[5] Pros And Cons Of Solid-State Batteries: Are Solid-State Batteries The Savior of The EV? - https://carbuzz.com/solid-state-batteries-pros-cons



=== SCORECARD ===
{
  "topic": "Solid-state batteries for electric vehicles: promise and obstacles",
  "checks": {
    "unique_domains": {
      "required": 3,
      "found_in_research": 5,
      "domains": [
        "atomfair.com",
        "carbuzz.com",
        "electricvehicletalks.com",
        "en.wikipedia.org",
        "next10insights.com"
      ],
      "cited_domains": [
        "atomfair.com",
        "carbuzz.com",
        "electricvehicletalks.com",
        "en.wikipedia.org"
      ],
      "pass": true
    },
    "short_quotes": {
      "required": 2,
      "verified_short_quotes": 4,
      "quotes_over_10_words": 0,
      "pass": true
    },
    "references_section": {
      "present": true,
      "agent_wrote_it_itself": true,
      "pass": true
    },
    "citations_resolve": {
      "cited": [
        2,
        3,
        4,
        5
      ],
      "unresolved": [],
      "pass": true
    },
    "paragraph_count": {
      "required": "5-8",
      "found": 6,
      "

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>